# Predict Customer Churn
## Score: .91408

In [19]:
N_FOLDS = 5
N_SEEDS = 3
OPTUNA_TRIALS = 50
USE_RF_ET = False
USE_ORIGINAL_DATA = True
TARGET_ENC_M = 20
BLEND_MODE = 'rank'
USE_ISOTONIC_CALIBRATION = True
USE_CV_TARGET_ENCODING = True
STACK_RANK_LINEAR = True

In [20]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = np.ones(len(train_combined))

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
print(f"Train: {len(X_full)}, Original: {USE_ORIGINAL_DATA}")

Train: 601237, Original: True


In [21]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

if USE_CV_TARGET_ENCODING:
    from sklearn.model_selection import StratifiedKFold
    cv_te = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for col in ['Contract', 'PaymentMethod', 'InternetService']:
        X_full[f'{col}_te'] = np.nan
        for train_idx, val_idx in cv_te.split(X_full, y_full):
            tr = X_full.iloc[train_idx].join(y_full.iloc[train_idx])
            global_mean = tr['Churn'].mean()
            agg = tr.groupby(col)['Churn'].agg(['mean', 'count'])
            smooth = (agg['mean'] * agg['count'] + global_mean * TARGET_ENC_M) / (agg['count'] + TARGET_ENC_M)
            X_full.loc[val_idx, f'{col}_te'] = X_full.loc[val_idx, col].map(smooth).fillna(global_mean).values
        comb = pd.DataFrame({col: X_full[col], '_y': y_full.values})
        agg_full = comb.groupby(col)['_y'].agg(['mean', 'count'])
        smooth_full = (agg_full['mean'] * agg_full['count'] + y_full.mean() * TARGET_ENC_M) / (agg_full['count'] + TARGET_ENC_M)
        X_test_raw[f'{col}_te'] = X_test_raw[col].map(smooth_full).fillna(y_full.mean())
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])
else:
    for col in ['Contract', 'PaymentMethod', 'InternetService']:
        X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=TARGET_ENC_M)
        X_full = X_full.drop(columns=[col])
        X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [22]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [23]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-09 15:41:28,840] A new study created in memory with name: no-name-8a77f327-7487-4238-a328-1a325df3871f


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-09 15:42:35,298] Trial 0 finished with value: 0.914833308911402 and parameters: {'n_estimators': 690, 'max_depth': 3, 'learning_rate': 0.11409171316207779, 'subsample': 0.7495849046030272, 'colsample_bytree': 0.7229844761029068, 'scale_pos_weight': 1.6518527408648662, 'reg_alpha': 0.18843912352486417, 'reg_lambda': 0.5661283740547899, 'min_child_weight': 7}. Best is trial 0 with value: 0.914833308911402.
[I 2026-03-09 15:44:01,141] Trial 1 finished with value: 0.914782427737905 and parameters: {'n_estimators': 627, 'max_depth': 6, 'learning_rate': 0.035703644899915066, 'subsample': 0.7853447465550732, 'colsample_bytree': 0.7478983533523936, 'scale_pos_weight': 3.758757150653791, 'reg_alpha': 0.019803470428706473, 'reg_lambda': 0.3419232473353824, 'min_child_weight': 3}. Best is trial 0 with value: 0.914833308911402.
[I 2026-03-09 15:44:40,010] Trial 2 finished with value: 0.9139411016650969 and parameters: {'n_estimators': 430, 'max_depth': 3, 'learning_rate': 0.048853003378

In [24]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-09 16:36:40,832] A new study created in memory with name: no-name-bceecc11-71ab-483c-944a-3c556bf50dc1


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-09 16:37:20,548] Trial 0 finished with value: 0.9153833905660527 and parameters: {'n_estimators': 531, 'max_depth': 4, 'learning_rate': 0.06571389636555917, 'subsample': 0.7250526231890899, 'colsample_bytree': 0.9432131448094285, 'scale_pos_weight': 2.325667004783959, 'reg_alpha': 4.92486489501191, 'reg_lambda': 1.0907567154275652, 'min_child_samples': 16}. Best is trial 0 with value: 0.9153833905660527.
[I 2026-03-09 16:37:53,610] Trial 1 finished with value: 0.9153580864292004 and parameters: {'n_estimators': 338, 'max_depth': 7, 'learning_rate': 0.09725392243806268, 'subsample': 0.7111609146055903, 'colsample_bytree': 0.6421496879080987, 'scale_pos_weight': 3.7913983727570955, 'reg_alpha': 1.8657899384785315, 'reg_lambda': 0.8710665635410672, 'min_child_samples': 17}. Best is trial 0 with value: 0.9153833905660527.
[I 2026-03-09 16:38:42,580] Trial 2 finished with value: 0.9152724751757315 and parameters: {'n_estimators': 574, 'max_depth': 6, 'learning_rate': 0.0742589556

In [25]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-09 17:12:15,639] A new study created in memory with name: no-name-514439f4-f6fd-4c85-8710-e4dc6e237637


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-09 17:13:13,762] Trial 0 finished with value: 0.914572477860332 and parameters: {'iterations': 383, 'depth': 3, 'learning_rate': 0.11705867127752126, 'subsample': 0.9089046983263418, 'colsample_bylevel': 0.6497982083370847, 'scale_pos_weight': 3.8315727075688186, 'l2_leaf_reg': 0.009055939735236777}. Best is trial 0 with value: 0.914572477860332.
[I 2026-03-09 17:14:07,371] Trial 1 finished with value: 0.9140034850193086 and parameters: {'iterations': 361, 'depth': 3, 'learning_rate': 0.07258382030163502, 'subsample': 0.7714403845736216, 'colsample_bylevel': 0.7721244281830696, 'scale_pos_weight': 3.182255573481646, 'l2_leaf_reg': 0.015674338148969073}. Best is trial 0 with value: 0.914572477860332.
[I 2026-03-09 17:16:51,212] Trial 2 finished with value: 0.9154402888651096 and parameters: {'iterations': 734, 'depth': 5, 'learning_rate': 0.10864641753706009, 'subsample': 0.8449125753579632, 'colsample_bylevel': 0.7590692330802085, 'scale_pos_weight': 3.9767521837333173, 'l2_

In [26]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize
from scipy.stats import rankdata

n_models = 3 if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

def to_rank_percentiles(arr):
    return (rankdata(arr) - 0.5) / len(arr)

oof_rank = np.column_stack([to_rank_percentiles(oof[:, i]) for i in range(n_models)])
oof_linear = oof.copy()

def neg_auc(w):
    blend = oof_blend_input @ w
    return -roc_auc_score(y_full, blend)

if STACK_RANK_LINEAR:
    oof_blend_input = oof_rank
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    w_rank = best_w
    oof_blend_input = oof_linear
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    w_linear = best_w
    blend_rank_oof = oof_rank @ w_rank
    blend_linear_oof = oof_linear @ w_linear
    best_auc, best_alpha = -1, 0.5
    for a in np.linspace(0, 1, 21):
        blended = a * blend_rank_oof + (1 - a) * blend_linear_oof
        auc = roc_auc_score(y_full, blended)
        if auc > best_auc:
            best_auc, best_alpha = auc, a
    blend_oof = best_alpha * blend_rank_oof + (1 - best_alpha) * blend_linear_oof
    blend_weights = best_alpha * w_rank + (1 - best_alpha) * w_linear
    stack_alpha = best_alpha
else:
    if BLEND_MODE == 'rank':
        oof_blend_input = oof_rank
    else:
        oof_blend_input = oof.copy()
    best_auc, best_w = -1, None
    for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
        result = minimize(neg_auc, x0=x0, method='SLSQP',
                          bounds=[(0, 1)]*n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        if -result.fun > best_auc:
            best_auc, best_w = -result.fun, result.x
    blend_weights = best_w
    blend_oof = oof_blend_input @ blend_weights
    stack_alpha = None

if USE_ISOTONIC_CALIBRATION:
    iso_calib = IsotonicRegression(out_of_bounds='clip')
    iso_calib.fit(blend_oof, y_full)
    blend_oof = np.clip(iso_calib.predict(blend_oof), 0, 1)

names = ['XGB','LGB','Cat'] + (['RF','ET'] if USE_RF_ET else [])
mode_str = 'rank+linear' if STACK_RANK_LINEAR else BLEND_MODE
print(f"Blend OOF AUC ({mode_str}): {roc_auc_score(y_full, blend_oof):.4f}")
print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")
if STACK_RANK_LINEAR:
    print(f"Stack alpha (rank): {stack_alpha:.3f}")

Blend OOF AUC (rank+linear): 0.9158
Weights: {'XGB': np.float64(0.3331), 'LGB': np.float64(0.3335), 'Cat': np.float64(0.3334)}
Stack alpha (rank): 1.000


In [27]:
def blend_test(tp):
    if STACK_RANK_LINEAR:
        blend_rank = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)]) @ w_rank
        blend_linear = tp @ w_linear
        preds = stack_alpha * blend_rank + (1 - stack_alpha) * blend_linear
    elif BLEND_MODE == 'rank':
        tp_input = np.column_stack([to_rank_percentiles(tp[:, i]) for i in range(n_models)])
        preds = tp_input @ blend_weights
    else:
        preds = tp @ blend_weights
    if USE_ISOTONIC_CALIBRATION:
        preds = np.clip(iso_calib.predict(preds), 0, 1)
    return preds

all_test_preds = [blend_test(test_preds)]
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(blend_test(tp))

final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.083434
1,594195,0.000451
2,594196,0.110062
3,594197,0.003876
4,594198,0.517662
